<a href="https://colab.research.google.com/github/sjasthi/Python-DS-Data-Science/blob/main/pandas/2_pandas_CRUD_Operations_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pandas - CRUD on a DataFrame

A DataFrame is a table. This notebook shows how to **C**reate, **R**ead, **U**pdate and **D**elete the things inside it.

| Section | What we work on | C | R | U | D |
|---|---|:-:|:-:|:-:|:-:|
| 1 | The **DataFrame** itself - different ways to create one | yes | yes | | |
| 2 | **Columns** | yes | yes | yes | yes |
| 3 | **Rows** | yes | yes | yes | yes |
| 4 | **Cells** | - | yes | yes | - |
| 5 | Gotchas and a cheat sheet | | | | |

**Cells can only be read and updated.** A cell exists as soon as its row and its column exist, so you can not create a cell or delete a cell. (To "clear" a cell you *update* it to `NaN`.)

**How to use this notebook**
- One small dataset (`students`) is used everywhere.
- Every section starts with `df = students.copy()`, so you can run any section on its own.
- Inside a section, cells build on each other - run them top to bottom.
- Missing-data handling and duplicate handling are covered in a separate data-cleaning notebook.

In [1]:
#@title Imports and the starting dataset
import pandas as pd
import numpy as np

print("pandas version:", pd.__version__)

# The dataset used throughout the notebook.
# Each section starts with a fresh copy:  df = students.copy()
students = pd.DataFrame({
    'ID':     [1, 2, 3, 4, 5, 6],
    'Name':   ['Anna', 'Bindu', 'Charlie', 'David', 'Eva', 'Farid'],
    'Score':  [85, 90, 78, 92, 66, 88],
    'Age':    [23, 25, 22, 14, 16, 30],
    'Course': ['Python 101', 'Python DS', 'Python 101', 'Python ML', 'Python DS', 'Python 101'],
})
display(students)

pandas version: 3.0.2


,ID,Name,Score,Age,Course
0,1,Anna,85,23,Python 101
1,2,Bindu,90,25,Python DS
2,3,Charlie,78,22,Python 101
3,4,David,92,14,Python ML
4,5,Eva,66,16,Python DS
5,6,Farid,88,30,Python 101


# 1. Creating a DataFrame

There are many ways to build a DataFrame. Pick the one that matches the shape of your data.

In [2]:
#@title 1.1 From a dictionary of lists (key = column name, list = column values)
data = {'ID': [1, 2, 3, 4],
        'Name': ['Anna', 'Bindu', 'Charlie', 'David'],
        'Score': [85, 90, 78, 92]}

df = pd.DataFrame(data)
display(df)
print("Type:", type(df))

,ID,Name,Score
0,1,Anna,85
1,2,Bindu,90
2,3,Charlie,78
3,4,David,92


Type: <class 'pandas.DataFrame'>


In [3]:
#@title 1.2 From a list of dictionaries (one dictionary per row)
# A key that is missing in a row becomes NaN.
records = [
    {'ID': 1, 'Name': 'Anna',    'Score': 85},
    {'ID': 2, 'Name': 'Bindu',   'Score': 90},
    {'ID': 3, 'Name': 'Charlie'},              # no 'Score' here -> NaN
]

df = pd.DataFrame(records)
display(df)

,ID,Name,Score
0,1,Anna,85.0
1,2,Bindu,90.0
2,3,Charlie,NaN


In [4]:
#@title 1.3 From a list of lists (or tuples) - give the column names with columns=
rows = [[1, 'Anna', 85],
        [2, 'Bindu', 90],
        [3, 'Charlie', 78]]
df_from_lists = pd.DataFrame(rows, columns=['ID', 'Name', 'Score'])
display(df_from_lists)

tuples = [(1, 'Anna', 85), (2, 'Bindu', 90), (3, 'Charlie', 78)]
df_from_tuples = pd.DataFrame(tuples, columns=['ID', 'Name', 'Score'])
display(df_from_tuples)

,ID,Name,Score
0,1,Anna,85
1,2,Bindu,90
2,3,Charlie,78


,ID,Name,Score
0,1,Anna,85
1,2,Bindu,90
2,3,Charlie,78


In [5]:
#@title 1.4 From a dictionary of Series (rows are matched up by index label)
scores = pd.Series({'Anna': 85, 'Bindu': 90, 'Charlie': 78})
ages   = pd.Series({'Anna': 23, 'Bindu': 25, 'David': 14})

# Charlie has no age and David has no score -> NaN
df = pd.DataFrame({'Score': scores, 'Age': ages})
display(df)

,Score,Age
Anna,85.0,23.0
Bindu,90.0,25.0
Charlie,78.0,NaN
David,NaN,14.0


In [6]:
#@title 1.5 From a NumPy array
arr = np.array([[85, 23],
                [90, 25],
                [78, 22]])

df = pd.DataFrame(arr, columns=['Score', 'Age'])
display(df)

,Score,Age
0,85,23
1,90,25
2,78,22


In [7]:
#@title 1.6 With your own row labels (index=) and from_dict(orient='index')
df = pd.DataFrame({'Score': [85, 90, 78]},
                  index=['Anna', 'Bindu', 'Charlie'])
display(df)

# Each key is a ROW label; each inner dictionary holds that row's values
by_student = {'Anna':  {'Score': 85, 'Age': 23},
              'Bindu': {'Score': 90, 'Age': 25}}
df2 = pd.DataFrame.from_dict(by_student, orient='index')
display(df2)

,Score
Anna,85
Bindu,90
Charlie,78


,Score,Age
Anna,85,23
Bindu,90,25


In [8]:
#@title 1.7 An empty DataFrame with columns, filled one row at a time
df = pd.DataFrame(columns=['ID', 'Name', 'Score'])
print("Empty?", df.empty, "| shape:", df.shape)

df.loc[0] = [1, 'Anna', 85]
df.loc[1] = [2, 'Bindu', 90]
display(df)

# Everything is stored as 'object' when built this way - fix the types
display(df.dtypes)
df = df.astype({'ID': int, 'Score': int})
display(df.dtypes)

Empty? True | shape: (0, 3)


,ID,Name,Score
0,1,Anna,85
1,2,Bindu,90


ID       int64
Name       str
Score    int64
dtype: object

ID       int64
Name       str
Score    int64
dtype: object

In [9]:
#@title 1.8 From another DataFrame - alias versus copy
original = pd.DataFrame({'Name': ['Anna', 'Bindu'], 'Score': [85, 90]})

alias     = original          # NOT a new DataFrame - just a second name for the same one
real_copy = original.copy()   # an independent DataFrame

alias.loc[0, 'Score'] = 100   # change through the alias...

print("original  (changed too!):")
display(original)
print("real_copy (unaffected):")
display(real_copy)

# A subset of another DataFrame - use .copy() when you plan to modify it
subset = students[['Name', 'Score']].copy()
display(subset.head(3))

original  (changed too!):


,Name,Score
0,Anna,100
1,Bindu,90


real_copy (unaffected):


,Name,Score
0,Anna,85
1,Bindu,90


,Name,Score
0,Anna,85
1,Bindu,90
2,Charlie,78


In [10]:
#@title 1.9 From files - CSV, Excel and JSON
# CSV from a URL
csv_url = 'https://raw.githubusercontent.com/sjasthi/Python-DS-Data-Science/main/datasets/titanic_data.csv'
df_csv = pd.read_csv(csv_url)
print("CSV   :", df_csv.shape)
display(df_csv.head())

# Excel from a URL  (needs the openpyxl package - it is pre-installed in Colab)
excel_url = 'https://raw.githubusercontent.com/sjasthi/Python-DS-Data-Science/main/datasets/titanic_data.xlsx'
df_excel = pd.read_excel(excel_url)
print("Excel :", df_excel.shape)
display(df_excel.head())

# JSON (here from a string; a file path or URL works the same way)
from io import StringIO
json_text = '[{"ID": 1, "Name": "Anna", "Score": 85}, {"ID": 2, "Name": "Bindu", "Score": 90}]'
df_json = pd.read_json(StringIO(json_text))
print("JSON  :", df_json.shape)
display(df_json)

# For a file on your own computer, use the file name instead of the URL:
#   pd.read_csv('my_file.csv')      pd.read_excel('my_file.xlsx')

CSV   : (1309, 12)


,Id,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Excel : (1309, 12)


,Id,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


JSON  : (2, 3)


,ID,Name,Score
0,1,Anna,85
1,2,Bindu,90


In [11]:
#@title 1.10 Quick checks right after creating a DataFrame
df = students.copy()

print("shape   :", df.shape)            # (rows, columns)
print("rows    :", len(df))
print("columns :", list(df.columns))
print("index   :", df.index)
print()
display(df.dtypes)
df.info()

shape   : (6, 5)
rows    : 6
columns : ['ID', 'Name', 'Score', 'Age', 'Course']
index   : RangeIndex(start=0, stop=6, step=1)



ID        int64
Name        str
Score     int64
Age       int64
Course      str
dtype: object

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   ID      6 non-null      int64
 1   Name    6 non-null      str  
 2   Score   6 non-null      int64
 3   Age     6 non-null      int64
 4   Course  6 non-null      str  
dtypes: int64(3), str(2)
memory usage: 372.0 bytes


# 2. Columns

Start from a fresh copy of the dataset. The cells in this section build on each other, top to bottom.

In [12]:
#@title Start section 2 with a fresh copy
df = students.copy()
display(df)

,ID,Name,Score,Age,Course
0,1,Anna,85,23,Python 101
1,2,Bindu,90,25,Python DS
2,3,Charlie,78,22,Python 101
3,4,David,92,14,Python ML
4,5,Eva,66,16,Python DS
5,6,Farid,88,30,Python 101


## 2.1 Create a column (C)

In [13]:
#@title Add a column - every row gets the same value
df['Term'] = 'Fall 2026'
display(df)

,ID,Name,Score,Age,Course,Term
0,1,Anna,85,23,Python 101,Fall 2026
1,2,Bindu,90,25,Python DS,Fall 2026
2,3,Charlie,78,22,Python 101,Fall 2026
3,4,David,92,14,Python ML,Fall 2026
4,5,Eva,66,16,Python DS,Fall 2026
5,6,Farid,88,30,Python 101,Fall 2026


In [14]:
#@title Add a column - a list with one value per row
df['Bonus_Points'] = [5, 0, 10, 0, 5, 2]
display(df)

# The list must be exactly as long as the DataFrame
try:
    df['Oops'] = [1, 2, 3]
except ValueError as e:
    print("ValueError:", e)

,ID,Name,Score,Age,Course,Term,Bonus_Points
0,1,Anna,85,23,Python 101,Fall 2026,5
1,2,Bindu,90,25,Python DS,Fall 2026,0
2,3,Charlie,78,22,Python 101,Fall 2026,10
3,4,David,92,14,Python ML,Fall 2026,0
4,5,Eva,66,16,Python DS,Fall 2026,5
5,6,Farid,88,30,Python 101,Fall 2026,2


ValueError: Length of values (3) does not match length of index (6)


In [15]:
#@title Add a column - calculated from other columns
df['Adjusted'] = df['Score'] + df['Bonus_Points']
df['Score_Pct'] = df['Score'] / 100
display(df[['Name', 'Score', 'Bonus_Points', 'Adjusted', 'Score_Pct']])

,Name,Score,Bonus_Points,Adjusted,Score_Pct
0,Anna,85,5,90,0.85
1,Bindu,90,0,90,0.90
2,Charlie,78,10,88,0.78
3,David,92,0,92,0.92
4,Eva,66,5,71,0.66
5,Farid,88,2,90,0.88


In [16]:
#@title Add a column - conditional value with np.where (if / else)
df['Result'] = np.where(df['Score'] >= 80, 'Pass', 'Fail')
display(df[['Name', 'Score', 'Result']])

,Name,Score,Result
0,Anna,85,Pass
1,Bindu,90,Pass
2,Charlie,78,Fail
3,David,92,Pass
4,Eva,66,Fail
5,Farid,88,Pass


In [17]:
#@title Add a column - recode another column with map() (function or dictionary)
def map_grade(score):
    if score >= 90:
        return 'A'
    elif score >= 80:
        return 'B'
    else:
        return 'C'

df['Grade'] = df['Score'].map(map_grade)            # map with a function

level = {'Python 101': 'Beginner', 'Python DS': 'Intermediate', 'Python ML': 'Advanced'}
df['Level'] = df['Course'].map(level)               # map with a dictionary

display(df[['Name', 'Score', 'Grade', 'Course', 'Level']])

,Name,Score,Grade,Course,Level
0,Anna,85,B,Python 101,Beginner
1,Bindu,90,A,Python DS,Intermediate
2,Charlie,78,C,Python 101,Beginner
3,David,92,A,Python ML,Advanced
4,Eva,66,C,Python DS,Intermediate
5,Farid,88,B,Python 101,Beginner


In [18]:
#@title Add a column - apply(): a lambda on one column, a function on whole rows
# 1) lambda on a single column
df['Age_Group'] = df['Age'].apply(lambda a: 'Minor' if a < 18 else 'Adult')

# 2) axis=1 passes a whole ROW to the function, so it can look at several columns
def final_grade(row):
    score = row['Score']
    if row['Course'] == 'Python ML':      # harder course gets a small boost
        score = score + 5
    if score >= 90:
        return 'A'
    elif score >= 80:
        return 'B'
    else:
        return 'C'

df['Final_Grade'] = df.apply(final_grade, axis=1)
display(df[['Name', 'Score', 'Course', 'Age_Group', 'Final_Grade']])

,Name,Score,Course,Age_Group,Final_Grade
0,Anna,85,Python 101,Adult,B
1,Bindu,90,Python DS,Adult,A
2,Charlie,78,Python 101,Adult,C
3,David,92,Python ML,Minor,A
4,Eva,66,Python DS,Minor,C
5,Farid,88,Python 101,Adult,B


In [19]:
#@title Add a column at a specific position - insert()
# insert(position, column_name, values)
df.insert(2, 'Email', df['Name'].str.lower() + '@example.com')
display(df[['ID', 'Name', 'Email', 'Score']])

,ID,Name,Email,Score
0,1,Anna,anna@example.com,85
1,2,Bindu,bindu@example.com,90
2,3,Charlie,charlie@example.com,78
3,4,David,david@example.com,92
4,5,Eva,eva@example.com,66
5,6,Farid,farid@example.com,88


In [20]:
#@title Add columns with assign() - returns a new DataFrame, handy for chaining
df = df.assign(Score_x2=df['Score'] * 2,
               Passed=lambda d: d['Score'] >= 80)   # lambda can use columns created just now
display(df[['Name', 'Score', 'Score_x2', 'Passed']])

,Name,Score,Score_x2,Passed
0,Anna,85,170,True
1,Bindu,90,180,True
2,Charlie,78,156,False
3,David,92,184,True
4,Eva,66,132,False
5,Farid,88,176,True


In [21]:
#@title Add columns from another DataFrame - concat with axis=1
extra = pd.DataFrame({'City': ['Minneapolis', 'Chennai', 'Austin', 'Pune', 'Oslo', 'Cairo']},
                     index=df.index)      # same index -> rows line up
df = pd.concat([df, extra], axis=1)
display(df[['Name', 'City']])

,Name,City
0,Anna,Minneapolis
1,Bindu,Chennai
2,Charlie,Austin
3,David,Pune
4,Eva,Oslo
5,Farid,Cairo


## 2.2 Read columns (R)

In [22]:
#@title One column (a Series) versus several columns (a DataFrame)
one  = df['Name']                # single brackets -> Series
many = df[['Name', 'Score']]     # double brackets (a list) -> DataFrame

print(type(one))
print(type(many))
display(many)

<class 'pandas.Series'>
<class 'pandas.DataFrame'>


,Name,Score
0,Anna,85
1,Bindu,90
2,Charlie,78
3,David,92
4,Eva,66
5,Farid,88


In [23]:
#@title Read columns by label (loc) and by position (iloc)
print("All column names:", list(df.columns))

display(df.loc[:, 'Name':'Score'].head(3))      # by label - the end label IS included
display(df.iloc[:, 0:3].head(3))                # by position - the end position is NOT included

display(df.select_dtypes(include='number').columns)   # only the numeric columns
display(df.select_dtypes(exclude='number').columns)   # everything that is not numeric (text, ...)

All column names: ['ID', 'Name', 'Email', 'Score', 'Age', 'Course', 'Term', 'Bonus_Points', 'Adjusted', 'Score_Pct', 'Result', 'Grade', 'Level', 'Age_Group', 'Final_Grade', 'Score_x2', 'Passed', 'City']


,Name,Email,Score
0,Anna,anna@example.com,85
1,Bindu,bindu@example.com,90
2,Charlie,charlie@example.com,78


,ID,Name,Email
0,1,Anna,anna@example.com
1,2,Bindu,bindu@example.com
2,3,Charlie,charlie@example.com


Index(['ID', 'Score', 'Age', 'Bonus_Points', 'Adjusted', 'Score_Pct',
       'Score_x2'],
      dtype='str')

Index(['Name', 'Email', 'Course', 'Term', 'Result', 'Grade', 'Level',
       'Age_Group', 'Final_Grade', 'Passed', 'City'],
      dtype='str')

## 2.3 Update columns (U)

In [24]:
#@title Update the values of a column
# NOTE: columns calculated earlier (Grade, Result, ...) do NOT update automatically.
df['Score'] = df['Score'] + 5                                   # whole column
df['Name'] = df['Name'].str.upper()                             # text method on a column
df['Course'] = df['Course'].replace({'Python 101': 'Python Basics'})   # swap specific values

def cap_score(x):
    return min(x, 100)
df['Score'] = df['Score'].apply(cap_score)                      # your own function

display(df[['Name', 'Score', 'Course', 'Grade']])

,Name,Score,Course,Grade
0,ANNA,90,Python Basics,B
1,BINDU,95,Python DS,A
2,CHARLIE,83,Python Basics,C
3,DAVID,97,Python ML,A
4,EVA,71,Python DS,C
5,FARID,93,Python Basics,B


In [25]:
#@title Rename columns
# rename() returns a NEW DataFrame - assign it back (df = ...) if you want to keep it
df_renamed = df.rename(columns={'Name': 'Student_Name'})
display(df_renamed.columns)

df_renamed = df.rename(columns={'Name': 'Student_Name', 'Score': 'Final_Score'})
display(df_renamed.columns)

# Rename ALL columns at once by assigning to .columns
df_lower = df.copy()
df_lower.columns = df_lower.columns.str.lower()
display(df_lower.columns)

Index(['ID', 'Student_Name', 'Email', 'Score', 'Age', 'Course', 'Term',
       'Bonus_Points', 'Adjusted', 'Score_Pct', 'Result', 'Grade', 'Level',
       'Age_Group', 'Final_Grade', 'Score_x2', 'Passed', 'City'],
      dtype='str')

Index(['ID', 'Student_Name', 'Email', 'Final_Score', 'Age', 'Course', 'Term',
       'Bonus_Points', 'Adjusted', 'Score_Pct', 'Result', 'Grade', 'Level',
       'Age_Group', 'Final_Grade', 'Score_x2', 'Passed', 'City'],
      dtype='str')

Index(['id', 'name', 'email', 'score', 'age', 'course', 'term', 'bonus_points',
       'adjusted', 'score_pct', 'result', 'grade', 'level', 'age_group',
       'final_grade', 'score_x2', 'passed', 'city'],
      dtype='str')

In [26]:
#@title Reorder columns
first = ['ID', 'Name', 'Course', 'Score']
others = [c for c in df.columns if c not in first]

df_reordered = df[first + others]
display(df_reordered.head(3))

,ID,Name,Course,Score,Email,Age,Term,Bonus_Points,Adjusted,Score_Pct,Result,Grade,Level,Age_Group,Final_Grade,Score_x2,Passed,City
0,1,ANNA,Python Basics,90,anna@example.com,23,Fall 2026,5,90,0.85,Pass,B,Beginner,Adult,B,170,True,Minneapolis
1,2,BINDU,Python DS,95,bindu@example.com,25,Fall 2026,0,90,0.90,Pass,A,Intermediate,Adult,A,180,True,Chennai
2,3,CHARLIE,Python Basics,83,charlie@example.com,22,Fall 2026,10,88,0.78,Fail,C,Beginner,Adult,C,156,False,Austin


In [27]:
#@title Change the data type of a column - astype() and to_numeric()
messy = pd.DataFrame({'ID':    ['1', '2', '3'],          # numbers stored as text
                      'Score': ['85.5', '90.3', 'abc'],  # one bad value
                      'Age':   [25, '30', 27]})          # mixed int / text
display(messy.dtypes)

messy['ID'] = messy['ID'].astype(int)                                # strict: fails on bad values
messy['Score'] = pd.to_numeric(messy['Score'], errors='coerce')     # forgiving: bad value -> NaN
messy['Age'] = pd.to_numeric(messy['Age'])

display(messy)
display(messy.dtypes)

ID          str
Score       str
Age      object
dtype: object

,ID,Score,Age
0,1,85.5,25
1,2,90.3,30
2,3,NaN,27


ID         int64
Score    float64
Age        int64
dtype: object

## 2.4 Delete columns (D)

In [28]:
#@title Delete columns - drop(), del and pop()
display(list(df.columns))

df = df.drop(columns=['Term'])                         # one column
df = df.drop(columns=['Bonus_Points', 'Score_Pct'])    # several columns
del df['Score_x2']                                     # del statement (changes df directly)
removed = df.pop('Passed')                             # removes the column AND hands it back

print("pop() returned:", removed.tolist())
display(list(df.columns))

# A column that does not exist raises a KeyError - errors='ignore' skips it silently
df = df.drop(columns=['NoSuchColumn'], errors='ignore')

['ID',
 'Name',
 'Email',
 'Score',
 'Age',
 'Course',
 'Term',
 'Bonus_Points',
 'Adjusted',
 'Score_Pct',
 'Result',
 'Grade',
 'Level',
 'Age_Group',
 'Final_Grade',
 'Score_x2',
 'Passed',
 'City']

pop() returned: [True, True, False, True, False, True]


['ID',
 'Name',
 'Email',
 'Score',
 'Age',
 'Course',
 'Adjusted',
 'Result',
 'Grade',
 'Level',
 'Age_Group',
 'Final_Grade',
 'City']

# 3. Rows

Start from a fresh copy. The cells in this section build on each other, top to bottom.

In [29]:
#@title Start section 3 with a fresh copy
df = students.copy()
display(df)

,ID,Name,Score,Age,Course
0,1,Anna,85,23,Python 101
1,2,Bindu,90,25,Python DS
2,3,Charlie,78,22,Python 101
3,4,David,92,14,Python ML
4,5,Eva,66,16,Python DS
5,6,Farid,88,30,Python 101


## 3.1 Create rows (C)

In [30]:
#@title Add one row at the end - loc[len(df)]
# len(df) is 6, so label 6 does not exist yet -> a new row is created.
# (This only works when the index is the default 0, 1, 2, ... )
df.loc[len(df)] = [7, 'Gita', 91, 27, 'Python DS']
display(df)

,ID,Name,Score,Age,Course
0,1,Anna,85,23,Python 101
1,2,Bindu,90,25,Python DS
2,3,Charlie,78,22,Python 101
3,4,David,92,14,Python ML
4,5,Eva,66,16,Python DS
5,6,Farid,88,30,Python 101
6,7,Gita,91,27,Python DS


In [31]:
#@title Add one row - concat() with a one-row DataFrame (the modern way)
# DataFrame.append() was removed in pandas 2.0 - old tutorials still show it.
new_row = {'ID': 8, 'Name': 'Hari', 'Score': 73, 'Age': 19, 'Course': 'Python 101'}

df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
display(df)

,ID,Name,Score,Age,Course
0,1,Anna,85,23,Python 101
1,2,Bindu,90,25,Python DS
2,3,Charlie,78,22,Python 101
3,4,David,92,14,Python ML
4,5,Eva,66,16,Python DS
5,6,Farid,88,30,Python 101
6,7,Gita,91,27,Python DS
7,8,Hari,73,19,Python 101


In [32]:
#@title Add many rows - concat() with a DataFrame of new rows
new_rows = pd.DataFrame({'ID':     [9, 10, 11],
                         'Name':   ['Isabel', 'Jack', 'Larry'],
                         'Score':  [88, 90, 95],
                         'Age':    [12, 13, 14],
                         'Course': ['Python DS', 'Python ML', 'Python 101']})

df = pd.concat([df, new_rows], ignore_index=True)     # ignore_index=True renumbers 0..n-1
display(df)

,ID,Name,Score,Age,Course
0,1,Anna,85,23,Python 101
1,2,Bindu,90,25,Python DS
2,3,Charlie,78,22,Python 101
3,4,David,92,14,Python ML
4,5,Eva,66,16,Python DS
5,6,Farid,88,30,Python 101
6,7,Gita,91,27,Python DS
7,8,Hari,73,19,Python 101
8,9,Isabel,88,12,Python DS
9,10,Jack,90,13,Python ML


In [33]:
#@title Insert a row at a specific position
pos = 2
row = pd.DataFrame([{'ID': 99, 'Name': 'Zed', 'Score': 70, 'Age': 21, 'Course': 'Python ML'}])

# top part + new row + bottom part
df = pd.concat([df.iloc[:pos], row, df.iloc[pos:]], ignore_index=True)
display(df.head(5))

,ID,Name,Score,Age,Course
0,1,Anna,85,23,Python 101
1,2,Bindu,90,25,Python DS
2,99,Zed,70,21,Python ML
3,3,Charlie,78,22,Python 101
4,4,David,92,14,Python ML


## 3.2 Read rows (R)

In [34]:
#@title First rows, last rows, random rows, how many
display(df.head(3))
display(df.tail(3))
display(df.sample(3, random_state=42))      # random_state makes the 'random' pick repeatable
print("Number of rows:", len(df), "| shape:", df.shape)

,ID,Name,Score,Age,Course
0,1,Anna,85,23,Python 101
1,2,Bindu,90,25,Python DS
2,99,Zed,70,21,Python ML


,ID,Name,Score,Age,Course
9,9,Isabel,88,12,Python DS
10,10,Jack,90,13,Python ML
11,11,Larry,95,14,Python 101


,ID,Name,Score,Age,Course
10,10,Jack,90,13,Python ML
9,9,Isabel,88,12,Python DS
0,1,Anna,85,23,Python 101


Number of rows: 12 | shape: (12, 5)


In [35]:
#@title Read rows by label (loc) and by position (iloc)
display(df.loc[2])              # the row labelled 2 -> a Series
display(df.loc[1:3])            # labels 1 to 3 - the end label IS included
display(df.iloc[1:3])           # positions 1 and 2 - the end position is NOT included
display(df.loc[[0, 2, 4], ['Name', 'Score']])   # chosen rows and chosen columns

ID               99
Name            Zed
Score            70
Age              21
Course    Python ML
Name: 2, dtype: object

,ID,Name,Score,Age,Course
1,2,Bindu,90,25,Python DS
2,99,Zed,70,21,Python ML
3,3,Charlie,78,22,Python 101


,ID,Name,Score,Age,Course
1,2,Bindu,90,25,Python DS
2,99,Zed,70,21,Python ML


,Name,Score
0,Anna,85
2,Zed,70
4,David,92


In [36]:
#@title Read rows that match a condition (boolean filtering)
print("Score >= 85")
display(df[df['Score'] >= 85])

print("Score >= 85 AND Age < 25   (each condition in its own parentheses)")
display(df[(df['Score'] >= 85) & (df['Age'] < 25)])

print("Course is Python ML OR Score < 70")
display(df[(df['Course'] == 'Python ML') | (df['Score'] < 70)])

print("Course is one of a list  - isin()")
display(df[df['Course'].isin(['Python DS', 'Python ML'])])

print("Same idea written as text  - query()")
display(df.query("Score >= 85 and Age < 25"))

Score >= 85


,ID,Name,Score,Age,Course
0,1,Anna,85,23,Python 101
1,2,Bindu,90,25,Python DS
4,4,David,92,14,Python ML
6,6,Farid,88,30,Python 101
7,7,Gita,91,27,Python DS
9,9,Isabel,88,12,Python DS
10,10,Jack,90,13,Python ML
11,11,Larry,95,14,Python 101


Score >= 85 AND Age < 25   (each condition in its own parentheses)


,ID,Name,Score,Age,Course
0,1,Anna,85,23,Python 101
4,4,David,92,14,Python ML
9,9,Isabel,88,12,Python DS
10,10,Jack,90,13,Python ML
11,11,Larry,95,14,Python 101


Course is Python ML OR Score < 70


,ID,Name,Score,Age,Course
2,99,Zed,70,21,Python ML
4,4,David,92,14,Python ML
5,5,Eva,66,16,Python DS
10,10,Jack,90,13,Python ML


Course is one of a list  - isin()


,ID,Name,Score,Age,Course
1,2,Bindu,90,25,Python DS
2,99,Zed,70,21,Python ML
4,4,David,92,14,Python ML
5,5,Eva,66,16,Python DS
7,7,Gita,91,27,Python DS
9,9,Isabel,88,12,Python DS
10,10,Jack,90,13,Python ML


Same idea written as text  - query()


,ID,Name,Score,Age,Course
0,1,Anna,85,23,Python 101
4,4,David,92,14,Python ML
9,9,Isabel,88,12,Python DS
10,10,Jack,90,13,Python ML
11,11,Larry,95,14,Python 101


In [37]:
#@title Read rows in a different order - sort_values()
top = df.sort_values(by='Score', ascending=False)      # returns a new, sorted DataFrame
display(top.head(3))

display(df.sort_values(by=['Course', 'Score'], ascending=[True, False]).head(5))

,ID,Name,Score,Age,Course
11,11,Larry,95,14,Python 101
4,4,David,92,14,Python ML
7,7,Gita,91,27,Python DS


,ID,Name,Score,Age,Course
11,11,Larry,95,14,Python 101
6,6,Farid,88,30,Python 101
0,1,Anna,85,23,Python 101
3,3,Charlie,78,22,Python 101
8,8,Hari,73,19,Python 101


## 3.3 Update rows (U)

In [38]:
#@title Update the rows that match a condition
# Change two columns for one student
df.loc[df['Name'] == 'Anna', ['Score', 'Age']] = [95, 24]

# Change one column for every row that matches
df.loc[df['Course'] == 'Python 101', 'Score'] += 2

display(df.head(6))

,ID,Name,Score,Age,Course
0,1,Anna,97,24,Python 101
1,2,Bindu,90,25,Python DS
2,99,Zed,70,21,Python ML
3,3,Charlie,80,22,Python 101
4,4,David,92,14,Python ML
5,5,Eva,66,16,Python DS


In [39]:
#@title Replace a whole row by its label
# one value for every column, in column order (row 2 is the 'Zed' row we inserted earlier)
df.loc[2] = [12, 'Zoe', 82, 20, 'Python DS']
display(df.head(4))

,ID,Name,Score,Age,Course
0,1,Anna,97,24,Python 101
1,2,Bindu,90,25,Python DS
2,12,Zoe,82,20,Python DS
3,3,Charlie,80,22,Python 101


## 3.4 Delete rows (D)

In [40]:
#@title Delete rows by index label - and why reset_index() matters
df = df.drop(index=2)            # one row
df = df.drop(index=[0, 4])       # several rows
display(df)                      # the index now has gaps: 1, 3, 5, 6, ...

# After deletions, a label and a position are no longer the same thing
print("df.loc[3]  -> label 3   :", df.loc[3, 'Name'])
print("df.iloc[3] -> position 3:", df.iloc[3]['Name'])

df = df.reset_index(drop=True)   # renumber 0..n-1 (drop=True: do not keep the old index as a column)
display(df)

,ID,Name,Score,Age,Course
1,2,Bindu,90,25,Python DS
3,3,Charlie,80,22,Python 101
5,5,Eva,66,16,Python DS
6,6,Farid,90,30,Python 101
7,7,Gita,91,27,Python DS
8,8,Hari,75,19,Python 101
9,9,Isabel,88,12,Python DS
10,10,Jack,90,13,Python ML
11,11,Larry,97,14,Python 101


df.loc[3]  -> label 3   : Charlie
df.iloc[3] -> position 3: Farid


,ID,Name,Score,Age,Course
0,2,Bindu,90,25,Python DS
1,3,Charlie,80,22,Python 101
2,5,Eva,66,16,Python DS
3,6,Farid,90,30,Python 101
4,7,Gita,91,27,Python DS
5,8,Hari,75,19,Python 101
6,9,Isabel,88,12,Python DS
7,10,Jack,90,13,Python ML
8,11,Larry,97,14,Python 101


In [41]:
#@title Delete rows that match a condition
# Way 1: find the labels of the rows to remove, then drop them
too_low = df[df['Score'] < 70].index
df = df.drop(index=too_low)

# Way 2: KEEP what you want. Here we remove rows where Age < 15 OR the course is Python ML.
df = df[~((df['Age'] < 15) | (df['Course'] == 'Python ML'))]

df = df.reset_index(drop=True)
display(df)

,ID,Name,Score,Age,Course
0,2,Bindu,90,25,Python DS
1,3,Charlie,80,22,Python 101
2,6,Farid,90,30,Python 101
3,7,Gita,91,27,Python DS
4,8,Hari,75,19,Python 101


In [42]:
#@title Delete ALL rows (the columns stay)
df_empty = df.iloc[0:0]            # zero rows, same columns
print("Rows left:", len(df_empty))
print("Columns  :", list(df_empty.columns))

Rows left: 0
Columns  : ['ID', 'Name', 'Score', 'Age', 'Course']


> Rows with missing values (`dropna`) and duplicate rows (`drop_duplicates`) are also row deletions. They are covered in the data-cleaning notebook.

# 4. Cells

A cell sits where one row and one column meet.

**Cells can only be read and updated.**
- You can not *create* a cell - it exists as soon as its row and column exist.
- You can not *delete* a cell - the table would no longer be rectangular. To "clear" a cell you *update* it to `NaN`.

In [43]:
#@title Start section 4 with a fresh copy
df = students.copy()
display(df)

,ID,Name,Score,Age,Course
0,1,Anna,85,23,Python 101
1,2,Bindu,90,25,Python DS
2,3,Charlie,78,22,Python 101
3,4,David,92,14,Python ML
4,5,Eva,66,16,Python DS
5,6,Farid,88,30,Python 101


## 4.1 Read cells (R)

In [44]:
#@title Read one cell - at, iat, loc, iloc
# at / loc use LABELS   |   iat / iloc use POSITIONS (row number, column number)
print("df.at[2, 'Name']    :", df.at[2, 'Name'])
print("df.iat[2, 1]        :", df.iat[2, 1])
print("df.loc[2, 'Name']   :", df.loc[2, 'Name'])
print("df.iloc[2, 1]       :", df.iloc[2, 1])

# at / iat are the fastest choice when you need exactly one cell

df.at[2, 'Name']    : Charlie
df.iat[2, 1]        : Charlie
df.loc[2, 'Name']   : Charlie
df.iloc[2, 1]       : Charlie


In [45]:
#@title Read a cell found by a condition, and a block of cells
# A condition returns a Series (there could be many matches) ...
match = df.loc[df['Name'] == 'Anna', 'Score']
print(type(match))

# ... take the single value out of it
print("Anna's score:", match.iloc[0])

# A block of cells: some rows x some columns
display(df.loc[1:3, ['Name', 'Score']])

<class 'pandas.Series'>
Anna's score: 85


,Name,Score
1,Bindu,90
2,Charlie,78
3,David,92


## 4.2 Update cells (U)

In [46]:
#@title Update one cell - at, iat, loc, iloc
df.at[2, 'Name'] = 'Charlie Brown'      # by labels
df.iat[0, 2] = 88                       # by positions: row 0, column 2 (Score)
df.loc[1, 'Score'] = 95                 # by labels
df.iloc[3, 3] = 15                      # by positions: row 3, column 3 (Age)

# A cell found by a condition
df.loc[df['Name'] == 'Eva', 'Score'] = 70

display(df)

,ID,Name,Score,Age,Course
0,1,Anna,88,23,Python 101
1,2,Bindu,95,25,Python DS
2,3,Charlie Brown,78,22,Python 101
3,4,David,92,15,Python ML
4,5,Eva,70,16,Python DS
5,6,Farid,88,30,Python 101


In [47]:
#@title Update a block of cells
# Same value for several cells
df.loc[df['Course'] == 'Python DS', 'Score'] += 2

# Apply a function to every cell of a block  (DataFrame.map needs pandas 2.1+)
numbers = df[['Score', 'Age']]
display(numbers.map(lambda x: x * 10))      # returns a new DataFrame

df[['Score', 'Age']] = numbers.map(lambda x: x + 1)   # assign it back to really update df
display(df)

,Score,Age
0,880,230
1,970,250
2,780,220
3,920,150
4,720,160
5,880,300


,ID,Name,Score,Age,Course
0,1,Anna,89,24,Python 101
1,2,Bindu,98,26,Python DS
2,3,Charlie Brown,79,23,Python 101
3,4,David,93,16,Python ML
4,5,Eva,73,17,Python DS
5,6,Farid,89,31,Python 101


In [48]:
#@title "Clear" a cell - update it to NaN
df.loc[4, 'Score'] = np.nan

display(df)
print("Missing values per column:")
display(df.isna().sum())

,ID,Name,Score,Age,Course
0,1,Anna,89.0,24,Python 101
1,2,Bindu,98.0,26,Python DS
2,3,Charlie Brown,79.0,23,Python 101
3,4,David,93.0,16,Python ML
4,5,Eva,NaN,17,Python DS
5,6,Farid,89.0,31,Python 101


Missing values per column:


ID        0
Name      0
Score     1
Age       0
Course    0
dtype: int64

In [49]:
#@title Gotcha - chained assignment silently updates nothing
import warnings

df_test = students.copy()

# WRONG: df[mask] makes a temporary copy, and the assignment lands on that copy
with warnings.catch_warnings():
    warnings.simplefilter('ignore')         # hide pandas' warning just for this demo
    df_test[df_test['Score'] > 80]['Score'] = 0
print("After the chained assignment:", df_test['Score'].tolist(), "  <- nothing changed")

# RIGHT: one loc[rows, column] does the whole update on df_test itself
df_test.loc[df_test['Score'] > 80, 'Score'] = 0
print("After loc                   :", df_test['Score'].tolist())

After the chained assignment: [85, 90, 78, 92, 66, 88]   <- nothing changed
After loc                   : [0, 0, 78, 0, 66, 0]


# 5. Gotchas and cheat sheet

**Gotchas**

1. **`inplace=True` versus assigning back.** Most methods (`drop`, `rename`, `sort_values`, `reset_index`, ...) return a *new* DataFrame. Either write `df = df.drop(...)` or use `inplace=True`, which changes `df` and returns `None`. Do not mix them: `df = df.drop(..., inplace=True)` sets `df` to `None`.
2. **Alias versus copy.** `b = a` is a second name for the same DataFrame. Use `b = a.copy()` for an independent one.
3. **Label versus position.** `loc` uses labels and *includes* the end of a slice. `iloc` uses positions and *excludes* the end. After deleting rows, labels and positions no longer match.
4. **Calculated columns are frozen.** A column created from `Score` does not change when `Score` changes later.
5. **Chained assignment.** `df[mask]['col'] = value` updates a temporary copy. Use `df.loc[mask, 'col'] = value`.
6. **Sizes must match.** A list assigned to a column needs one value per row; a list assigned to a row needs one value per column.
7. **`reset_index(drop=True)` after deleting rows** if you want 0, 1, 2, ... again.

In [50]:
#@title inplace=True returns None
df = students.copy()

result = df.drop(columns=['Age'], inplace=True)
print("Return value of drop(inplace=True):", result)
print("But df itself changed:", list(df.columns))

Return value of drop(inplace=True): None
But df itself changed: ['ID', 'Name', 'Score', 'Course']


**Cheat sheet**

| | **C**reate | **R**ead | **U**pdate | **D**elete |
|---|---|---|---|---|
| **DataFrame** | `pd.DataFrame(dict / list of dicts / list of lists / array)`, `read_csv`, `read_excel`, `read_json`, `.copy()` | `head()`, `shape`, `columns`, `dtypes`, `info()` | - | - |
| **Column** | `df['c'] = ...`, `insert()`, `assign()`, `concat(axis=1)` | `df['c']`, `df[['a','b']]`, `loc[:, ...]`, `iloc[:, ...]`, `select_dtypes()` | `df['c'] = ...`, `map()`, `apply()`, `replace()`, `rename()`, `astype()`, reorder with `df[list]` | `drop(columns=...)`, `del df['c']`, `pop('c')` |
| **Row** | `loc[len(df)] = ...`, `concat([...])`, insert with `iloc` slices + `concat` | `head()`, `tail()`, `sample()`, `loc`, `iloc`, `df[condition]`, `query()`, `sort_values()` | `loc[condition, cols] = ...`, `loc[label] = [...]` | `drop(index=...)`, `df[~condition]`, `reset_index(drop=True)` |
| **Cell** | not possible | `at`, `iat`, `loc[r, c]`, `iloc[r, c]` | `at`, `iat`, `loc[r, c] = ...`, `iloc[r, c] = ...` | not possible (set to `np.nan` = update) |